# 03/02 — Disease signature + per-region rescue

With only 1 WT mouse, per-region WT-anchored t-tests are not valid. Pivot:

1. **Disease signature (global, once).** Whole-tissue PBS vs WT pseudobulk (n=7 vs n=1). Pooled-variance fallback in `pseudobulk_lfc(min_n=1)`. Rank disease-up / disease-down genes. Cross-check against literature PIG/DAM lists.
2. **Per-region rescue (well-powered).** BRI vs PBS per region (n=3 vs n=2-7). Welch on log-CPM.
3. **T2' sign concordance.** Of disease-up genes, fraction *lowered* by BRICHOS per region (binomial vs 0.5).
4. **T3' rescue slope.** Regress per-region LFC_BRI-vs-PBS on global LFC_disease over disease-DEGs. Slope < 0 = rescue.

Outputs `regional_rescue_stats.tsv`, `disease_signature.tsv`.

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'           # change to 'library_id' if obs uses that
REGION_KEY    = 'anatomical_region'   # adjust to your obs column for regions
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('count layer :', COUNT_LAYER)


In [ ]:
from utils.attenuation import (
    pseudobulk_lfc, signed_rescue, rescue_slope
)

counts = pd.read_csv(TBL / 'pseudobulk_counts.tsv',
                     sep='\t', index_col=0)
meta   = pd.read_csv(TBL / 'pseudobulk_meta.tsv',
                     sep='\t', index_col=0)
regions = sorted(meta['region'].dropna().unique())
print(len(regions), 'regions')


### 1. Disease signature — whole-tissue PBS vs WT

Sum across regions per (sample). With n=1 WT we use `min_n=1`: pooled variance from PBS replicates. **Significance values here are inflated** — use the LFC ranking and treat padj as descriptive.

In [ ]:
global_counts = counts.copy()
global_counts['__sample__'] = meta['sample']
global_counts = global_counts.groupby('__sample__').sum()
global_meta = (meta.groupby('sample')
               .agg(treatment=('treatment','first'))
               .reset_index().set_index('sample'))
global_meta.index.name = None
global_meta = global_meta.rename_axis(None)
# emulate the (sample, region) row-id format expected by helpers
global_counts.index = global_counts.index.astype(str)
global_meta.index   = global_meta.index.astype(str)
global_meta['region'] = '__all__'
global_meta['sample'] = global_meta.index

disease = pseudobulk_lfc(global_counts, global_meta,
                         group_a='PBS', group_b='WT',
                         region='__all__', min_n=1)
disease_df = disease.to_frame()
disease_df.to_csv(TBL / 'disease_signature.tsv', sep='\t')
print(f'  n_a={disease.n_a}, n_b={disease.n_b}')
disease_df.sort_values('lfc', ascending=False).head()


### Pick disease gene set

Two ways, pick one or report both:

* **Data-driven**: top genes by |LFC| in PBS-vs-WT (descriptive padj filter).
* **Literature**: known PIG / DAM gene lists.

Default: union, with provenance tag.

In [ ]:
# Data-driven set
lfc_thresh, padj_thresh = 0.5, 0.1
data_up = disease_df.index[(disease_df.lfc >  lfc_thresh) &
                            (disease_df.padj < padj_thresh)]
data_dn = disease_df.index[(disease_df.lfc < -lfc_thresh) &
                            (disease_df.padj < padj_thresh)]

# Literature PIG (Chen et al, Cell 2020) — full curated list
# from utils/gene_sets.py.
from utils.gene_sets import PIG_CHEN2020, filter_to_var
lit_up = filter_to_var(PIG_CHEN2020, disease_df.index)
print(f'PIG genes in disease table: {len(lit_up)} / {len(PIG_CHEN2020)}')

disease_up = pd.Index(set(list(data_up) + lit_up))
disease_dn = pd.Index(data_dn)
print(f'data-driven up: {len(data_up)}, dn: {len(data_dn)}')
print(f'literature PIG present: {len(lit_up)}')
print(f'union disease-up: {len(disease_up)}')


### 2. Per-region BRI vs PBS LFC

In [ ]:
rescue_lfc = {}
for region in regions:
    try:
        res = pseudobulk_lfc(counts, meta,
                             group_a='BRICHOS', group_b='PBS',
                             region=region, min_n=2)
    except ValueError as e:
        print(f'  {region}: skip ({e})')
        continue
    rescue_lfc[region] = res
    print(f'  {region}: n_BRI={res.n_a}, n_PBS={res.n_b}, '
          f'n_genes={len(res.lfc)}')


### 3. + 4. Per-region rescue stats

In [ ]:
rows = []
for region, res in rescue_lfc.items():
    sc_res = signed_rescue(res.lfc, disease_up, disease_dn)
    sl_res = rescue_slope(disease_df.lfc, res.lfc,
                          sig_mask=list(disease_up),
                          n_boot=1000)
    rows.append(dict(
        region=region,
        n_BRI=res.n_a, n_PBS=res.n_b,
        n_signature=sc_res['n'],
        frac_concordant=sc_res['frac_concordant'],
        binom_p=sc_res['binom_p'],
        wilcoxon_p=sc_res['wilcoxon_p'],
        median_signed_lfc=sc_res['median_signed_lfc'],
        slope=sl_res['slope'],
        slope_ci_low=sl_res['ci_low'],
        slope_ci_high=sl_res['ci_high'],
    ))
stats_df = (pd.DataFrame(rows).set_index('region')
            .sort_values('slope'))
stats_df.to_csv(TBL / 'regional_rescue_stats.tsv', sep='\t')
stats_df


### Plot — rescue slope per region

Negative slope = treatment opposes disease. Bootstrap CI shown.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 0.4 * len(stats_df) + 1.0))
y = np.arange(len(stats_df))
ax.errorbar(stats_df['slope'], y,
            xerr=[stats_df['slope'] - stats_df['slope_ci_low'],
                  stats_df['slope_ci_high'] - stats_df['slope']],
            fmt='o', color='#222', ecolor='#888', capsize=2)
ax.axvline(0, color='#bbb', linewidth=0.6, linestyle=':')
ax.axvline(-1, color='#d62728', linewidth=0.6, linestyle='--',
           alpha=0.4)
ax.set_yticks(y); ax.set_yticklabels(stats_df.index, fontsize=8)
ax.set_xlabel('rescue slope (LFC$_{BRI-PBS}$ on LFC$_{disease}$)')
ax.set_title('0 = no effect   |   -1 = full rescue', fontsize=8,
             loc='left', color='#555')
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / 'regional_rescue_slope.svg', bbox_inches='tight')
fig.savefig(FIG / 'regional_rescue_slope.png',
            bbox_inches='tight', dpi=200)
plt.show()


### Plot — sign concordance fraction

Of disease-signature genes, fraction moved in the rescue direction by BRICHOS. 0.5 is the chance level.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 0.4 * len(stats_df) + 1.0))
y = np.arange(len(stats_df))
ax.barh(y, stats_df['frac_concordant'] - 0.5, left=0.5,
        color='#1f77b4', alpha=0.7, edgecolor='#222')
for i, p in enumerate(stats_df['binom_p'].values):
    star = '***' if p < 1e-3 else ('**' if p < 1e-2 else
           ('*' if p < 5e-2 else ''))
    ax.text(stats_df['frac_concordant'].iloc[i] + 0.01, i,
            star, fontsize=8, va='center')
ax.axvline(0.5, color='#bbb', linewidth=0.6, linestyle=':')
ax.set_yticks(y); ax.set_yticklabels(stats_df.index, fontsize=8)
ax.set_xlim(0.3, 1.0)
ax.set_xlabel('fraction of signature genes moved by BRICHOS '
              'in rescue direction')
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / 'regional_sign_concordance.svg',
            bbox_inches='tight')
fig.savefig(FIG / 'regional_sign_concordance.png',
            bbox_inches='tight', dpi=200)
plt.show()
